In [5]:
import ast
import numpy as np
import pandas as pd
import torch

CSV_PATH = "data/climate_ttc/climate_2014_2023_final_with_embeddings_lag_3.csv"
MAX_ROWS = 10  # set to None or -1 to use all rows
EMBED_PREFIX = "embedding_text_lag"

# Load and parse embedding columns
df = pd.read_csv(CSV_PATH)
if MAX_ROWS is not None and MAX_ROWS >= 0:
    df = df.head(MAX_ROWS)

emb_cols = [c for c in df.columns if c.startswith(EMBED_PREFIX)]
if not emb_cols:
    raise ValueError(f"No embedding_text_lag* cols found in {CSV_PATH}")

for col in emb_cols:
    df[col] = df[col].apply(ast.literal_eval)

def concat_embeddings(row):
    return np.concatenate([np.asarray(row[c], dtype=np.float32) for c in emb_cols])

X_text = np.stack([concat_embeddings(row) for _, row in df[emb_cols].iterrows()])

# Compute attention
X = torch.tensor(X_text, dtype=torch.float32)
X_norm = torch.nn.functional.normalize(X, dim=1)
sim = X_norm @ X_norm.T
attn = torch.softmax(sim, dim=-1)

# Summary
attn_np = attn.detach().cpu().numpy()
diag = np.diag(attn_np)
off_diag = attn_np[~np.eye(attn_np.shape[0], dtype=bool)]
print(f"Attn shape: {attn_np.shape}")
print(f"Diag mean={diag.mean():.4f}, std={diag.std():.4f}")
print(f"Off-diag mean={off_diag.mean():.4f}, std={off_diag.std():.4f}")

row0 = attn_np[0]
top5 = row0.argsort()[::-1][:5]
print("Top-5 attention targets for row 0 (idx: weight):")
for idx in top5:
    print(f"  {idx}: {row0[idx]:.4f}")


Attn shape: (10, 10)
Diag mean=0.1431, std=0.0081
Off-diag mean=0.0952, std=0.0111
Top-5 attention targets for row 0 (idx: weight):
  0: 0.1650
  1: 0.1291
  2: 0.1064
  9: 0.0885
  8: 0.0866
